<a href="https://colab.research.google.com/github/scalabrinig/cdProjetoAplicadoIV/blob/master/projeto/cd_projeto_aplicado_IV_doc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

  <img src="https://raw.githubusercontent.com/scalabrinig/cdProjetoAplicadoIV/d093146488f56dfcf0ef286bcee8efe0e71b9c76/figuras/mackenzie_logo.jpg" width="25%" align="right"/>

# **APPLIED PROJECT IV - Data Science - 2026/02**


# **Project Documentation**

---

# **FORECASTING ARBOVIRUS CASES AND HOSPITAL DEMAND FROM CLIMATE AND SEASONALITY IN RIO DE JANEIRO**
---

# **Abstract**

This project proposes a predictive analytics product to support hospital management during arbovirus outbreaks (dengue, Zika and chikungunya) in the municipality of Rio de Janeiro. Time series techniques are applied to forecast the weekly evolution of cases and hospitalizations from epidemiological and climate data, and these forecasts are translated into an operational hospital alert level (Normal, Attention, Critical), calibrated against installed bed capacity and being discussed with professionals from the Rio de Janeiro Municipal Health Department, to ensure the rigor and social impact of the proposed solution.


# **Introduction**

Time series analysis and modeling are a strategic tool in public health for understanding and fighting infectious diseases. In Rio de Janeiro, the recurring epidemic behavior of arboviruses such as dengue, Zika and chikungunya puts heavy pressure on the public and private hospital network: outbreaks cause sudden peaks in visits and admissions, overloading the health infrastructure and making it hard to plan beds, medical staff and critical supplies in advance.

The proliferation of the *Aedes aegypti* mosquito and the transmission dynamics in the region are directly related to weather variables (temperature and accumulated rainfall) and to region-specific seasonal patterns. Predictive models that link lagged climate drivers to the evolution of cases and hospitalizations enable a shift from reactive hospital management to a preventive, optimized approach, which is the core motivation of this project.

**General objective:** to develop and validate a two-layer predictive analytics product:

1. Multivariate time series modeling to estimate the incidence of arbovirus cases and hospitalizations in Rio de Janeiro based on climate factors and seasonality
2. An operational classification model that assigns a hospital demand alert level (Normal, Attention, Critical), by combining the time series forecasts with the installed capacity of SUS (Brazil's public health system) beds.

**Specific objectives:**

1. Consolidate and harmonize historical epidemiological and weather records for Rio de Janeiro at the Epidemiological Week scale (2015–2025)
2. Characterize the structural components of the time series (trend, seasonality, cycles and autocorrelation)
3. Implement and compare statistical and deep learning forecasting algorithms (SARIMAX and LSTM/RNN)
4. Build and calibrate the hospital risk classification model with technical support from SUS-RJ specialists
5. Publish the results as interactive analytical documentation on GitHub

**Data:** multivariate time series organized by the official Epidemiological Week calendar, covering 2015 to 2025 continuously for Rio de Janeiro. Sources:

- **InfoDengue (Fiocruz/FGV):** weekly history of estimated and confirmed dengue, Zika and chikungunya notifications, incidence rate and regional alert levels
- **DATASUS (SINAN/SIH-SUS):** disease notifications and Hospital Admission Authorizations (AIH) for arboviruses in the public network
- **INMET/Open-Meteo:** weekly mean temperature, maximum temperature and accumulated rainfall
- **CNES:** installed capacity of ward and ICU beds, used as the reference for hospital alert levels

**Rationale:** although Brazil already has consolidated epidemiological monitoring platforms, such as InfoDengue itself, this project differs in three ways:

1. **Focus on operational hospital demand**, not only on notification/transmission risk in the population, by modeling how infections turn into hospitalizations and real bed needs
2. **Two-layer architecture**, combining quantitative time series forecasting (SARIMAX and LSTM) with a functional hospital alert classification layer
3. **Practical validation with SUS-RJ specialists**, with formal technical calibration cycles throughout development.

The project is aligned with the UN 2030 Agenda, directly addressing:

- **SDG 3 - Good Health and Well-being:** by providing predictive intelligence tools to support hospital management and rapid response to epidemics.
- **SDG 11 - Sustainable Cities and Communities:** by strengthening the resilience of urban health systems to climate variability and seasonal outbreaks in large cities.


# **Theoretical Background**

Predictive modeling of vector-borne diseases with time series is an established field, with approaches that have evolved from classical statistical models to hybrid techniques combining statistics and deep learning.

**1. Classical statistical models (SARIMA/SARIMAX)**

- The **SARIMA/SARIMAX** model (*Seasonal Autoregressive Integrated Moving Average with eXogenous variables*), based on Box and Jenkins (1970), remains a reference for epidemiological forecasting.
- Martinez and Silva (2011), using dengue data from Campinas (SP), validated its applicability to case series with a strong seasonal component.
- **Advantage:** interpretable coefficients and low computational cost.
- **Limitation:** it assumes mostly linear relationships between the series and the exogenous variables, which may not capture the full complexity of climate–epidemiological phenomena.

**2. Deep learning models (LSTM)**

- **LSTM** networks (*Long Short-Term Memory*), proposed by Hochreiter and Schmidhuber (1997), have been applied to the same problem because they can capture nonlinear and long-term dependencies.
- Sahd et al. (2026) directly compared SARIMAX and LSTM for dengue forecasting in Londrina (PR), using temperature and humidity as exogenous variables.
- **Result:** SARIMAX was more stable, while LSTM, despite more volatile forecasts, proved promising. This motivates comparing both techniques in this project as well, instead of adopting a single approach a priori.

**3. Studies on Rio de Janeiro and hospital demand**

- Recent literature has already evaluated statistical and machine learning models for dengue forecasting specifically in Rio de Janeiro, supporting the methodological feasibility of the chosen geographic scope.
- There are also studies that specifically forecast dengue **hospitalizations** in Brazil from climate data, accounting for real reporting delays, a practical challenge also faced in this project given the use of AIH/SIH-SUS data.
- *(The full references for these two studies are being verified and will be added in the next bibliography update.)*

**4. Decomposition and transformation techniques**

- **STL decomposition** (*Seasonal-Trend decomposition using Loess*), proposed by Cleveland et al. (1990), was used to characterize the series because it is more robust to seasonality with varying amplitude, which matters here, since arbovirus outbreaks do not have the same seasonal intensity every year.
- Variance stabilization of the case series used the **Box-Cox transformation** (Box and Cox, 1964), with the optimal parameter estimated by Guerrero's method (1993).

**5. What sets this project apart**

Most related literature focuses on forecasting **case incidence**, not **operational bed demand**. This project differs by:

- Combining time series forecasting (SARIMAX/LSTM) with a hospital risk classification layer
- Calibrating this classification against installed bed capacity (CNES)
- Validating the results with SUS-RJ professionals, turning the epidemiological forecast into a direct operational decision for hospital management

# **EDA and Data Preprocessing**

A. Main points:

**Data and scope**
- Weekly series from 2015 to 2025 for the municipality of Rio de Janeiro: estimated dengue, chikungunya and Zika cases (InfoDengue) and daily temperature and rainfall (Open-Meteo), aggregated by epidemiological week.
- In the period where all three series exist (2015–Feb/2024), dengue accounts for about 60% of cases, chikungunya 28% and Zika 12%.
- The InfoDengue API returns no Zika data for the municipality after February 2024; therefore, the target variable is the sum of dengue and chikungunya, and Zika appears only in the descriptive analysis.

**Quality and preparation**
- Week alignment fix: climate is now aggregated Sunday to Saturday and labeled by the epidemiological week's starting Sunday, as in InfoDengue (previously there was a hidden lag of about one week).
- Internal gaps imputed by time interpolation; no synthetic data is used in the results.
- Box-Cox transformation with λ ≈ −0.02, practically logarithmic; therefore `y = log(1 + cases)` was adopted.

**Series characteristics**
- ADF test: stationary series (p < 0.05).
- STL decomposition with a 52-week period: strong trend (F_T ≈ 0.75) and moderate annual seasonality (F_S ≈ 0.41), which justifies the seasonal term S = 52.
- Cross-correlation: temperature has the highest correlation with cases at about an 11-week lag (r ≈ 0.45); rainfall has a weak correlation (r ≈ 0.16).

**Hospital data (Stage 4)**
- SIH-SUS admissions and length-of-stay days by residence in the municipality, CNES SUS beds by specialty and notified dengue cases by Planning Area (SMS-Rio Municipal Tabnet), plus 2022 population by neighborhood (Data.Rio), mapped to the 10 Planning Areas.

# **Models**

**Case forecasting** ([Stage 3 notebook](https://github.com/CarlaDutra25/dengue-hospital-demand-rio/blob/main/eda_sarimax.ipynb))
- Models compared: ARIMAX with Fourier terms (with and without climate), SARIMAX(1,0,1)(1,0,0)₅₂ with lagged climate, and LSTM (optional).
- Baselines: persistence (repeat the last observed value) and seasonal naive (repeat the same week of the previous year).
- Rolling-origin *walk-forward* validation over three test years (2023, 2024 and 2025): for each year, climate lags, standardization, model order and parameters are set using only previous years, with no information leakage. Forecasts are made 1 to 4 weeks ahead, with 80% prediction intervals.
- Climate enters only with a lag greater than or equal to the forecast horizon, so it is available at forecast time.

**Hospital demand, alert and Planning Areas** ([Stage 4 notebook](https://github.com/CarlaDutra25/dengue-hospital-demand-rio/blob/main/demand_alert.ipynb))
- Forecast cases are converted into admissions and occupied beds using the hospitalization rate per case and the mean length of stay from SIH-SUS in the 12 months before the forecast (Little's law: beds ≈ weekly admissions × length of stay ÷ 7).
- Forecast occupancy is compared with CNES SUS general medicine and pediatric beds to produce the Normal / Attention / Critical alert (provisional thresholds of 1% and 2% of beds, to be calibrated with SMS-RJ).
- The municipal forecast is disaggregated to the 10 Planning Areas by each area's share of notified cases (*top-down*), and the ability to identify the highest-incidence areas in advance is evaluated.

> **Note:** the Stage 3 and Stage 4 notebooks are documented in Portuguese (original course language); code, tables and figures are self-explanatory.

# **Results**

Table 2 shows the performance of the best case forecasting model compared with persistence, aggregated over the three test years.

**Table 2 — Weekly case forecasting, walk-forward validation 2023–2025**

| Horizon | WAPE SARIMAX | WAPE Persistence | SARIMAX gain over persistence (MAE) |
|---|---|---|---|
| 1 week | 20.4% | 20.2% | −1.3% |
| 2 weeks | 34.1% | 34.8% | +1.8% |
| 3 weeks | 45.2% | 48.1% | +6.0% |
| 4 weeks | 54.8% | 60.3% | +9.2% |

- Seasonal SARIMAX was the most stable model across years; the Fourier models varied widely (best in 2023, worst in 2024).
- Lagged climate brought no additional forecasting gain.
- The 80% prediction intervals had empirical coverage between 82% and 93%.
- Epidemic peaks were forecast, on average, about 4 weeks late.

Table 3 summarizes the results of the hospital and territorial layer, compared with actual SIH-SUS and Municipal Tabnet data.

**Table 3 — Hospital demand, alert and Planning Areas (forecasts 4 weeks ahead, 2023–2025)**

| Indicator | SARIMAX | Persistence |
|---|---|---|
| Monthly occupied beds error (MAE) | 4.2 beds | 4.6 beds |
| Monthly occupied beds error (WAPE) | 44.2% | 48.7% |
| Monthly alert level accuracy | 93.9% | 90.9% |
| Top-3 priority Planning Areas hit rate | 80% (long-run share) / 72% (recent) | — |

- At the peak of the 2024 epidemic (March), arboviruses occupied on average about 61 SUS beds, roughly 3.1% of the city's general medicine and pediatric beds.
- The two months classified as Critical in 2024 (February and March) were flagged 4 weeks in advance.

# **Discussion and Conclusion**

**Key findings**
- Classical statistical models track the case series well but anticipate little: the gain over persistence only appears 3 to 4 weeks ahead and is modest (≈ 9%).
- The operational value lies in turning the forecast into a decision: reliable prediction intervals, alert levels and bed estimates validated against real hospital data.
- Even at the peak of the 2024 epidemic, average occupancy was around 3% of the city's SUS general medicine and pediatric beds. Hospital pressure is concentrated in specific areas and weeks, not a citywide capacity shortage; this reinforces the importance of the Planning Area view.

**Limitations**
- The alert was evaluated on few months of real pressure (February to May 2024).
- Municipal hospitalization rate and length of stay applied equally to all Planning Areas; beds and actual occupancy by area are not available in tabulated public data.
- Area shares based on notified dengue cases from 2023 onward; Zika excluded from the target; provisional alert thresholds.
- The SIH group "other arthropod-borne viral fevers" includes arboviruses other than chikungunya.

**Next steps**
- Threshold calibration and validation with the technical team of the Rio de Janeiro Municipal Health Department.
- Adding beds and occupancy by Planning Area and a prototype regulation-support layer based on demand × supply: where forecast demand will arise and where capacity is available.

# **Presentation**

The presentation video link will be included in the final deliverable (Stage 4).

# **References**

BOX, G. E. P.; JENKINS, G. M. **Time Series Analysis: Forecasting and Control**. San Francisco: Holden-Day, 1970.

BOX, G. E. P.; COX, D. R. An analysis of transformations. **Journal of the Royal Statistical Society, Series B, Statistical Methodology**, v. 26, n. 2, p. 211-252, 1964.

CLEVELAND, R. B. et al. STL: A seasonal-trend decomposition procedure based on Loess. **Journal of Official Statistics**, v. 6, n. 1, p. 3-73, 1990.

GUERRERO, V. M. Time-series analysis supported by power transformations. **Journal of Forecasting**, v. 12, n. 1, p. 37-48, 1993.

HOCHREITER, S.; SCHMIDHUBER, J. Long Short-Term Memory. **Neural Computation**, v. 9, n. 8, p. 1735-1780, 1997.

SAHD, C. S.; LIZZI, E. A. da S.; BRESSAN, G. M.; VILAS-BOAS, L. A. Forecasting Using SARIMAX Model and LSTM Approach for Arboviruses Using Exogenous Climatic Variables in Predicting Dengue Incidence. In: **Bioinformatics and Computational Biology (X-Meeting 2025)**. Cham: Springer, 2026. p. 214-226.

BRASIL. Ministério da Saúde. Departamento de Informática do SUS (DATASUS). **Doenças e Agravos de Notificação – 2007 em diante (SINAN)**. Brasília, DF: Ministério da Saúde, 2026. Available at: https://datasus.saude.gov.br/.

FUNDAÇÃO OSWALDO CRUZ. **Observatório Clima e Saúde: Mudanças climáticas e crescimento da dengue no Brasil**. Rio de Janeiro: Fiocruz, 2026. Available at: https://climaesaude.icict.fiocruz.br/.

INFODENGUE. **Serviços e Documentação da API**. Rio de Janeiro: Fundação Oswaldo Cruz / Fundação Getulio Vargas, 2026. Available at: https://info.dengue.mat.br/.